# Housing Application Approval

## Dataset

We use the ACSMobility dataset from the Folktables benchmark, derived from the U.S. Census American Community Survey (ACS).

The task is a binary classification problem that predicts whether a person changed residence in the past year (1) or did not move (0). Residential mobility is commonly used as a proxy for housing stability, since frequent or involuntary moves are often associated with housing insecurity, evictions, and barriers to housing approval.

The input features include demographic and socioeconomic variables such as age, education, marital status, employment characteristics, and income-related attributes.

The dataset also contains sensitive attributes (race and sex), which are used to evaluate fairness in housing-related decision-making.

## Race (RAC1P)

The variable RAC1P represents self-identified race in the ACS data.

- 1: White alone
- 2: Black or African American alone
- 3: American Indian or Alaska Native alone
- 4: Alaska Native alone
- 5: American Indian alone
- 6: Asian alone
- 7: Native Hawaiian or Other Pacific Islander alone
- 8: Some other race alone
- 9: Two or more races

### Sex (SEX)

The variable SEX is binary in the ACS:

- 1: Male
- 2: Female

## Models

We train diverse models on the same dataset:

### Accuracy-first model

This model uses all available features, including sensitive attributes, and is optimized purely for predictive performance (accuracy and ROC AUC). It represents a standard automated ML pipeline without fairness constraints.

### Fairness-aware model

This model excludes sensitive attributes (race and sex) from the feature set to reduce direct discrimination. While this may slightly reduce predictive accuracy, it aims to lower disparities between protected groups.

### Accuracy and fairness model (reweighting)

Rather than removing sensitive attributes, the model adopts a reweighting strategy that increases the influence of protected groups during training. This allows the model to retain access to sensitive information while mitigating disparities, resulting in more balance between accuracy and fairness.

### Adaptive reweighting model

The model applies an adaptive reweighting strategy in which the training importance of protected groups is gradually increased by different weight factors. By modifying these weights, the model gives more influence to underrepresented groups during training, allowing us to observe how fairness improves as the weights increase, and to analyze the balance between predictive accuracy and fairness.

### Fairness-aware scoring model

Instead of selecting the model that maximizes predictive performance alone, the model ranks candidate models using a joint objective that penalizes unfairness. Concretely, the model computes a composite score AUC − λ·fairness_gap, where fairness_gap is measured using demographic parity gaps across race and sex. 

## Data download

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from folktables import ACSDataSource, ACSMobility


# Load data
data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
acs_data = data_source.get_data(states=["CA"], download=True)

# Define task
task = ACSMobility

# Get filtered dataframe used by the task
acs_df, y, _ = task.df_to_pandas(acs_data)

# Build modeling dataframe
df = acs_df[task.features].copy()
df["target"] = y

# Add sensitive attributes
df["race"] = acs_df["RAC1P"].to_numpy()
df["sex"]  = acs_df["SEX"].to_numpy()

# Define feature columns
feature_names = [c for c in df.columns if c not in ["target", "race", "sex"]]

X = df[feature_names]
y = df["target"].astype(int)

## Train/Test split

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Model 1: Accuracy-first

In [12]:
model_acc = LogisticRegression(max_iter=2000)
model_acc.fit(X_train_s, y_train)

y_pred = model_acc.predict(X_test_s)
y_prob = model_acc.predict_proba(X_test_s)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.7662558612390555
ROC AUC: 0.6583981266877864


## Fairness (baseline)

In [13]:
test_df = df.loc[X_test.index, ["race", "sex"]].copy()
test_df["prob"] = y_prob

print("\nAvg predicted housing rejection by race:")
print(test_df.groupby("race")["prob"].mean())

print("\nAvg predicted housing rejection by sex:")
print(test_df.groupby("sex")["prob"].mean())

race_gap = test_df.groupby("race")["prob"].mean().max() - test_df.groupby("race")["prob"].mean().min()
sex_gap  = test_df.groupby("sex")["prob"].mean().max()  - test_df.groupby("sex")["prob"].mean().min()

print("\nDemographic Parity gap (race):", race_gap)
print("Demographic Parity gap (sex):", sex_gap)


Avg predicted housing rejection by race:
race
1.0    0.757438
2.0    0.747875
3.0    0.775632
5.0    0.743930
6.0    0.755815
7.0    0.771702
8.0    0.807744
9.0    0.784587
Name: prob, dtype: float64

Avg predicted housing rejection by sex:
sex
1.0    0.764158
2.0    0.767617
Name: prob, dtype: float64

Demographic Parity gap (race): 0.06381434507456951
Demographic Parity gap (sex): 0.0034596847782091267


# Model 2: Fairness-aware model

In [5]:
fair_features = [f for f in feature_names if f not in ["RAC1P", "SEX"]]

X_fair = df[fair_features]
y = df["target"].astype(int)

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.3, random_state=42, stratify=y
)

scaler_f = StandardScaler()
X_train_f_s = scaler_f.fit_transform(X_train_f)
X_test_f_s = scaler_f.transform(X_test_f)

model_fair = LogisticRegression(max_iter=2000)
model_fair.fit(X_train_f_s, y_train_f)

y_prob_f = model_fair.predict_proba(X_test_f_s)[:, 1]

## Fairness-aware evaluation

In [16]:
test_df_f = df.loc[X_test_f.index, ["race", "sex"]].copy()
test_df_f["prob"] = y_prob_f

print("\nAvg predicted housing rejection by race:")
print(test_df_f.groupby("race")["prob"].mean())

print("\nAvg predicted housing rejection by sex:")
print(test_df_f.groupby("sex")["prob"].mean())

race_gap_f = test_df_f.groupby("race")["prob"].mean().max() - test_df_f.groupby("race")["prob"].mean().min()
sex_gap_f  = test_df_f.groupby("sex")["prob"].mean().max()  - test_df_f.groupby("sex")["prob"].mean().min()

print("\nDemographic Parity gap (race):", race_gap_f)
print("Demographic Parity gap (sex):", sex_gap_f)


Avg predicted housing rejection by race:
race
1.0    0.765768
2.0    0.752223
3.0    0.776673
5.0    0.736558
6.0    0.747271
7.0    0.757448
8.0    0.792592
9.0    0.763459
Name: prob, dtype: float64

Avg predicted housing rejection by sex:
sex
1.0    0.762117
2.0    0.770137
Name: prob, dtype: float64

Demographic Parity gap (race): 0.05603401501019645
Demographic Parity gap (sex): 0.008020138840541535


# Model 3: Prioritizes Accuracy and Fairness without eliminating the sensitive attributes

In [7]:
# Features and target
feature_names = [c for c in df.columns if c not in ["target", "race", "sex"]]
X = df[feature_names]
y = df["target"].astype(int)

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Defining weights according to weights (reweighting)

In [17]:
train_df = df.loc[X_train.index, ["race", "sex"]]

# Initialize weights
sample_weights = np.ones(len(train_df))

# Increase weight for protected groups
sample_weights[train_df["sex"] == 2] *= 1.5
sample_weights[train_df["race"] != 1] *= 1.5

model_housing_balanced = LogisticRegression(max_iter=2000)

model_housing_balanced.fit(
    X_train_s,
    y_train,
    sample_weight=sample_weights
)

y_pred = model_housing_balanced.predict(X_test_s)
y_prob = model_housing_balanced.predict_proba(X_test_s)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.76696128470061
ROC AUC: 0.6578152453643222


## Fairness Evaluation

In [18]:
test_df = df.loc[X_test.index, ["race", "sex"]].copy()
test_df["prob"] = y_prob

print("\nAvg predicted housing rejection by race:")
print(test_df.groupby("race")["prob"].mean())

print("\nAvg predicted housing rejection by sex:")
print(test_df.groupby("sex")["prob"].mean())

race_gap = test_df.groupby("race")["prob"].mean().max() - test_df.groupby("race")["prob"].mean().min()
sex_gap  = test_df.groupby("sex")["prob"].mean().max()  - test_df.groupby("sex")["prob"].mean().min()

print("\nDemographic Parity gap (race):", race_gap)
print("Demographic Parity gap (sex):", sex_gap)


Avg predicted housing rejection by race:
race
1.0    0.757144
2.0    0.747451
3.0    0.775191
5.0    0.743924
6.0    0.754259
7.0    0.772228
8.0    0.807655
9.0    0.785421
Name: prob, dtype: float64

Avg predicted housing rejection by sex:
sex
1.0    0.763677
2.0    0.767301
Name: prob, dtype: float64

Demographic Parity gap (race): 0.06373156600025243
Demographic Parity gap (sex): 0.0036232896918243496


# Model 4: Adaptive weighting

In [19]:
feature_names = [c for c in df.columns if c not in ["target", "race", "sex"]]
X = df[feature_names]
y = df["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

train_groups = df.loc[X_train.index, ["race", "sex"]]
test_groups  = df.loc[X_test.index,  ["race", "sex"]]

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

weight_grid = [1.0, 1.1, 1.25, 1.5, 2.0]

results = []

for w in weight_grid:
    sample_weights = np.ones(len(train_groups), dtype=float)

    # Protected groups
    is_female = (train_groups["sex"] == 2)
    is_nonwhite = (train_groups["race"] != 1)

    sample_weights[is_female] *= w
    sample_weights[is_nonwhite] *= w

    model = LogisticRegression(max_iter=2000)
    model.fit(X_train_s, y_train, sample_weight=sample_weights)

    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    # Fairness metrics
    test_df = test_groups.copy()
    test_df["prob"] = y_prob

    avg_by_race = test_df.groupby("race")["prob"].mean()
    avg_by_sex  = test_df.groupby("sex")["prob"].mean()

    race_dp_gap = avg_by_race.max() - avg_by_race.min()
    sex_dp_gap  = avg_by_sex.max()  - avg_by_sex.min()

    # Fairness % for females (reference = male)
    ref_male = avg_by_sex.loc[1.0] if 1.0 in avg_by_sex.index else avg_by_sex.iloc[0]
    female_fairness_pct = (
        (avg_by_sex.loc[2.0] / ref_male) * 100
        if 2.0 in avg_by_sex.index else np.nan
    )

    results.append({
        "weight_multiplier": w,
        "accuracy": acc,
        "roc_auc": auc,
        "race_dp_gap": race_dp_gap,
        "sex_dp_gap": sex_dp_gap,
        "female_fairness_%": female_fairness_pct
    })

# Print results
print("\nAdaptive reweighting results:")
for r in results:
    print(
        f"w={r['weight_multiplier']}: "
        f"acc={r['accuracy']:.4f}, auc={r['roc_auc']:.4f}, "
        f"race_DP_gap={r['race_dp_gap']:.4f}, sex_DP_gap={r['sex_dp_gap']:.4f}, "
        f"female_fairness%={r['female_fairness_%']:.2f}%"
    )


Adaptive reweighting results:
w=1.0: acc=0.7663, auc=0.6584, race_DP_gap=0.0638, sex_DP_gap=0.0035, female_fairness%=100.45%
w=1.1: acc=0.7665, auc=0.6583, race_DP_gap=0.0638, sex_DP_gap=0.0035, female_fairness%=100.46%
w=1.25: acc=0.7669, auc=0.6581, race_DP_gap=0.0638, sex_DP_gap=0.0035, female_fairness%=100.46%
w=1.5: acc=0.7670, auc=0.6578, race_DP_gap=0.0637, sex_DP_gap=0.0036, female_fairness%=100.47%
w=2.0: acc=0.7667, auc=0.6571, race_DP_gap=0.0637, sex_DP_gap=0.0037, female_fairness%=100.49%


## Model 5: Fairness-aware scoring model

In [11]:
# 1) Prepare data 
feature_names = [c for c in df.columns if c not in ["target", "race", "sex"]]
X = df[feature_names]
y = df["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

train_groups = df.loc[X_train.index, ["race", "sex"]]
test_groups  = df.loc[X_test.index,  ["race", "sex"]]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# 2) Candidate models (different reweighting levels)
weight_grid = [1.0, 1.1, 1.25, 1.5, 2.0]

candidates = []

for w in weight_grid:
    sample_weights = np.ones(len(train_groups), dtype=float)

    # Protected groups
    sample_weights[train_groups["sex"] == 2] *= w        # female
    sample_weights[train_groups["race"] != 1] *= w       # non-white

    model = LogisticRegression(max_iter=2000)
    model.fit(X_train_s, y_train, sample_weight=sample_weights)

    y_prob = model.predict_proba(X_test_s)[:, 1]
    auc = roc_auc_score(y_test, y_prob)

    # Fairness metrics
    test_df = test_groups.copy()
    test_df["prob"] = y_prob

    avg_by_race = test_df.groupby("race")["prob"].mean()
    avg_by_sex  = test_df.groupby("sex")["prob"].mean()

    race_dp_gap = float(avg_by_race.max() - avg_by_race.min())
    sex_dp_gap  = float(avg_by_sex.max()  - avg_by_sex.min())

    candidates.append({
        "w": w,
        "auc": auc,
        "race_dp_gap": race_dp_gap,
        "sex_dp_gap": sex_dp_gap
    })

# 3) Fairness-aware scoring and selection
alpha = 1.0  # race weight
beta  = 1.0  # sex weight

lambda_grid = [0.0, 0.5, 1.0, 2.0, 5.0]

print("\nFairness-aware scoring :")
print("score = AUC − λ · (race_gap + sex_gap)\n")

best_overall = None

for lam in lambda_grid:
    best = None
    for c in candidates:
        fairness_penalty = alpha * c["race_dp_gap"] + beta * c["sex_dp_gap"]
        score = c["auc"] - lam * fairness_penalty

        if best is None or score > best["score"]:
            best = {**c, "lambda": lam, "score": score}

    print(
        f"lambda={lam}: BEST w={best['w']} | "
        f"AUC={best['auc']:.4f} | race_gap={best['race_dp_gap']:.4f} | sex_gap={best['sex_dp_gap']:.4f} | "
        f"score={best['score']:.4f}"
    )

    if best_overall is None or best["score"] > best_overall["score"]:
        best_overall = best

print("\nSelected model (highest combined score):")
print(
    f"lambda={best_overall['lambda']} | w={best_overall['w']} | "
    f"AUC={best_overall['auc']:.4f} | race_gap={best_overall['race_dp_gap']:.4f} | sex_gap={best_overall['sex_dp_gap']:.4f} | "
    f"score={best_overall['score']:.4f}"
)


Fairness-aware scoring :
score = AUC − λ · (race_gap + sex_gap)

lambda=0.0: BEST w=1.0 | AUC=0.6584 | race_gap=0.0638 | sex_gap=0.0035 | score=0.6584
lambda=0.5: BEST w=1.0 | AUC=0.6584 | race_gap=0.0638 | sex_gap=0.0035 | score=0.6248
lambda=1.0: BEST w=1.0 | AUC=0.6584 | race_gap=0.0638 | sex_gap=0.0035 | score=0.5911
lambda=2.0: BEST w=1.0 | AUC=0.6584 | race_gap=0.0638 | sex_gap=0.0035 | score=0.5239
lambda=5.0: BEST w=1.0 | AUC=0.6584 | race_gap=0.0638 | sex_gap=0.0035 | score=0.3220

Selected model (highest combined score):
lambda=0.0 | w=1.0 | AUC=0.6584 | race_gap=0.0638 | sex_gap=0.0035 | score=0.6584
